<a href="https://colab.research.google.com/github/khanakshah27/GuardianAI/blob/model/guardianai_crimemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision opencv-python-headless scikit-learn kagglehub

In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import cv2
import kagglehub
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:

dataset_path = kagglehub.dataset_download("mission-ai/crimeucfdataset")
print("Dataset path:", dataset_path)

100%|██████████| 32.9G/32.9G [19:07<00:00, 30.8MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/mission-ai/crimeucfdataset/versions/2


In [4]:

def load_video_paths(root_dir):
    video_data = []
    print("Scanning for videos...")

    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith((".mp4", ".avi")):
                full_path = os.path.join(root, file)

                if "Normal" in full_path:
                    label = "no crime"
                else:
                    label = "crime"

                if os.path.getsize(full_path) > 0:
                    video_data.append((full_path, label))

    return video_data

all_videos = load_video_paths(dataset_path)

all_normal = [x for x in all_videos if x[1] == "no crime"]
all_crime = [x for x in all_videos if x[1] == "crime"]

print(f"Total Normal found: {len(all_normal)}")
print(f"Total Crime found: {len(all_crime)}")


random.shuffle(all_normal)
random.shuffle(all_crime)

n_select = 100
selected_normal = all_normal[:n_select]
selected_crime = all_crime[:n_select]


labeled_videos = selected_normal + selected_crime
random.shuffle(labeled_videos)

print(f"\n--- DATASET BALANCING ---")
print(f"Selected Normal: {len(selected_normal)}")
print(f"Selected Crime: {len(selected_crime)}")
print(f"Total Final Dataset: {len(labeled_videos)}")

Scanning for videos...
Total Normal found: 300
Total Crime found: 800

--- DATASET BALANCING ---
Selected Normal: 100
Selected Crime: 100
Total Final Dataset: 200


In [5]:

label_map = {"no crime": 0, "crime": 1}


mean = [0.43216, 0.394666, 0.37645]
std = [0.22803, 0.22145, 0.216989]

class CrimeVideoDataset(Dataset):
    def __init__(self, video_data, clip_len=16, num_clips=5, size=112):
        self.data = video_data
        self.clip_len = clip_len
        self.num_clips = num_clips
        self.size = size
        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std)
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        cap = cv2.VideoCapture(path)

        if not cap.isOpened():
            print(f"Warning: Could not open video file {path}")
            return torch.zeros(self.num_clips, 3, self.clip_len, self.size, self.size), torch.tensor(label_map[label])

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames == 0:
            print(f"Warning: Video file {path} has 0 frames.")
            cap.release()
            return torch.zeros(self.num_clips, 3, self.clip_len, self.size, self.size), torch.tensor(label_map[label])


        max_start_frame = max(0, total_frames - self.clip_len)


        if total_frames < self.clip_len:
            starts = np.array([0])
            effective_num_clips = 1
        else:
            starts = np.linspace(0, max_start_frame, self.num_clips, dtype=int)
            effective_num_clips = self.num_clips

        clips_data = []
        for start_frame_idx in starts:
            frames_for_clip = []

            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame_idx)

            for _ in range(self.clip_len):
                ret, frame = cap.read()
                if not ret or frame is None:
                    frames_for_clip.append(torch.zeros(3, self.size, self.size))
                else:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames_for_clip.append(self.transform(frame))

            if len(frames_for_clip) < self.clip_len:
                padding = [torch.zeros(3, self.size, self.size)] * (self.clip_len - len(frames_for_clip))
                frames_for_clip.extend(padding)

            clips_data.append(torch.stack(frames_for_clip).permute(1, 0, 2, 3)) # C, T, H, W

        cap.release()

        if not clips_data:
            return torch.zeros(self.num_clips, 3, self.clip_len, self.size, self.size), torch.tensor(label_map[label])


        if len(clips_data) < self.num_clips:
            padding_clips = [torch.zeros(3, self.clip_len, self.size, self.size)] * (self.num_clips - len(clips_data))
            clips_data.extend(padding_clips)

        return torch.stack(clips_data), torch.tensor(label_map[label])

In [6]:

X_paths = [x[0] for x in labeled_videos]
y_labels = [label_map[x[1]] for x in labeled_videos]
X_train, X_test, y_train, y_test = train_test_split(
    labeled_videos, y_labels, test_size=0.2, stratify=y_labels, random_state=42
)

print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples: {len(X_test)}")

Training Samples: 160
Testing Samples: 40


In [7]:
train_dataset = CrimeVideoDataset(X_train)
test_dataset = CrimeVideoDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

In [8]:

r3d = r3d_18(weights=R3D_18_Weights.DEFAULT)

r3d.fc = nn.Identity()
r3d = r3d.to(device)
r3d.eval()

print("Feature extractor ready.")

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 152MB/s]


Feature extractor ready.


In [9]:
def extract_features(model, loader):
    features_list = []
    labels_list = []

    print("Extracting features...")
    with torch.no_grad():
        for i, (clips, labels) in enumerate(loader):

            bs, num_clips, C, T, H, W = clips.shape

            clips = clips.view(-1, C, T, H, W).to(device)

            outputs = model(clips)
            outputs = outputs.view(bs, -1).cpu().numpy()

            features_list.append(outputs)
            labels_list.append(labels.numpy())

            if (i+1) % 5 == 0:
                print(f"Processed batch {i+1}/{len(loader)}")

    return np.vstack(features_list), np.hstack(labels_list)

In [10]:

import psutil
import torch
n_cores = psutil.cpu_count(logical=False)
optimal_workers = min(2, n_cores)


use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

if not use_cuda:
    print("WARNING: Running on CPU. This will be slow. Enable GPU in Runtime > Change runtime type.")

print(f"Configuring datasets: 5 clips/video | Batch Size: 16 | Workers: {optimal_workers}")

train_dataset = CrimeVideoDataset(X_train, clip_len=16, num_clips=5)
test_dataset = CrimeVideoDataset(X_test, clip_len=16, num_clips=5)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=optimal_workers,
    pin_memory=use_cuda
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=optimal_workers,
    pin_memory=use_cuda
)

print("\n--- Processing Training Data ---")
try:
    X_train_features, y_train_labels = extract_features(r3d, train_loader)

    print("\n--- Processing Testing Data ---")
    X_test_features, y_test_labels = extract_features(r3d, test_loader)

    print(f"\nSuccess! Feature Shape: {X_train_features.shape}")

except NameError:
    print("Error: 'extract_features_fast' is not defined. Please run Cell 9.")
except RuntimeError as e:
    print(f"Runtime Error: {e}")
    print("Tip: If this is an Out of Memory error, try reducing batch_size to 8.")

Configuring datasets: 5 clips/video | Batch Size: 16 | Workers: 1

--- Processing Training Data ---
Extracting features...
Processed batch 5/10
Processed batch 10/10

--- Processing Testing Data ---
Extracting features...

Success! Feature Shape: (160, 2560)


In [ ]:
print("Training Random Forest Classifier...")

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_features, y_train_labels)

print("Training Complete.")

Training Random Forest Classifier...
Training Complete.


In [ ]:
print("Evaluating on Test Set...")
y_pred = rf_classifier.predict(X_test_features)

accuracy = accuracy_score(y_test_labels, y_pred)
cm = confusion_matrix(y_test_labels, y_pred)

print("\n========== FINAL RESULTS ==========")
print(f"Accuracy: {accuracy*100:.2f}%")
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred, target_names=['No Crime', 'Crime']))

if accuracy >= 0.80:
    print("\nSUCCESS: Accuracy goal of 80%+ met!")
else:
    print("\nResult below 80%.")

Evaluating on Test Set...

========== FINAL RESULTS ==========
Accuracy: 82.50%
Confusion Matrix:
[[17  3]
 [ 4 16]]

Classification Report:
              precision    recall  f1-score   support

    No Crime       0.81      0.85      0.83        20
       Crime       0.84      0.80      0.82        20

    accuracy                           0.82        40
   macro avg       0.83      0.82      0.82        40
weighted avg       0.83      0.82      0.82        40


SUCCESS: Accuracy goal of 80%+ met!


In [ ]:

import joblib
import tarfile
import torch
from google.colab import files

print("Saving model artifacts...")

joblib.dump(rf_classifier, 'crime_rf_model.pkl')

torch.save(r3d.state_dict(), 'model_weights.pth')

print("Compressing into model.tar.gz...")
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("crime_rf_model.pkl")
    tar.add("model_weights.pth")

print("Downloading files...")
files.download('model_weights.pth')

Saving model artifacts...
Compressing into model.tar.gz...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('crime_rf_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>